In [ ]:
import requests
import pandas as pd
from datetime import datetime

# 1. Configurar tus credenciales de Last.fm
API_KEY = "9235c93d8e4a547a26bf34df456b86d2"
# Método de la API para obtener el Top de canciones globales
METODO = "chart.gettoptracks"
URL = "http://ws.audioscrobbler.com/2.0/"

# 2. Definir los parámetros de la consulta (Pedimos el formato JSON y el Top 50)
parametros = {
    'method': METODO,
    'api_key': API_KEY,
    'format': 'json',
    'limit': 50
}

print("🚀 Conectando a la API de Last.fm y descargando el Top 50 Global...")

# 3. Realizar la petición HTTP a internet
respuesta = requests.get(URL, params=parametros)

if respuesta.status_code == 200:
    data_cruda = respuesta.json()
    lista_canciones = data_cruda['tracks']['track']

    # 4. Procesar y estructurar la materia prima
    fecha_hoy = datetime.now().strftime('%Y-%m-%d')
    materia_prima = []

    for indice, track in enumerate(lista_canciones):
        # Last.fm no da un ID único para todo, así que creamos una clave limpia
        # combinando el nombre del artista y la canción (quitando espacios raros)
        song_name = track['name']
        artist_name = track['artist']['name']
        song_id = f"{artist_name}_{song_name}".replace(" ", "_").lower()[:50]
        artist_id = artist_name.replace(" ", "_").lower()[:50]

        fila = {
            "date_key": fecha_hoy,
            "rank_position": indice + 1,
            "song_id": song_id,
            "song_name": song_name,
            "artist_id": artist_id,
            "artist_name": artist_name,
            "playcount_daily": int(track['playcount']),
            "listeners_daily": int(track['listeners']),
            "song_url": track['url'],
            "artist_url": track['artist']['url']
        }
        materia_prima.append(fila)

    # 5. Convertir a un DataFrame de Pandas
    df_global_charts = pd.DataFrame(materia_prima)
    print("✅ ¡Materia prima extraída y estructurada con éxito!")

else:
    print(f"❌ Error al conectar con la API. Código de estado: {respuesta.status_code}")

# Mostrar las primeras 5 filas de la materia prima generada
df_global_charts.head()

🚀 Conectando a la API de Last.fm y descargando el Top 50 Global...
✅ ¡Materia prima extraída y estructurada con éxito!


,date_key,rank_position,song_id,song_name,artist_id,artist_name,playcount_daily,listeners_daily,song_url,artist_url
0,2026-06-21,1,olivia_rodrigo_stupid_song,stupid song,olivia_rodrigo,Olivia Rodrigo,3757975,503496,https://www.last.fm/music/Olivia+Rodrigo/_/stu...,https://www.last.fm/music/Olivia+Rodrigo
1,2026-06-21,2,olivia_rodrigo_the_cure,the cure,olivia_rodrigo,Olivia Rodrigo,8321580,654587,https://www.last.fm/music/Olivia+Rodrigo/_/the...,https://www.last.fm/music/Olivia+Rodrigo
2,2026-06-21,3,olivia_rodrigo_drop_dead,drop dead,olivia_rodrigo,Olivia Rodrigo,12653859,786042,https://www.last.fm/music/Olivia+Rodrigo/_/dro...,https://www.last.fm/music/Olivia+Rodrigo
3,2026-06-21,4,olivia_rodrigo_honeybee,honeybee,olivia_rodrigo,Olivia Rodrigo,2667347,444306,https://www.last.fm/music/Olivia+Rodrigo/_/hon...,https://www.last.fm/music/Olivia+Rodrigo
4,2026-06-21,5,olivia_rodrigo_maggots_for_brains,maggots for brains,olivia_rodrigo,Olivia Rodrigo,2633117,426260,https://www.last.fm/music/Olivia+Rodrigo/_/mag...,https://www.last.fm/music/Olivia+Rodrigo


In [ ]:
from google.colab import files

print("⚙️ Iniciando normalización de datos para el Modelo Dimensional...")

# 1. Crear Dim_Songs (Solo datos descriptivos de la canción, eliminando duplicados si los hubiera)
dim_songs = df_global_charts[['song_id', 'song_name', 'song_url']].drop_duplicates().copy()

# 2. Crear Dim_Artists (Solo datos descriptivos del artista, eliminando duplicados)
dim_artists = df_global_charts[['artist_id', 'artist_name', 'artist_url']].drop_duplicates().copy()

# 3. Crear Fact_Global_Charts (Métricas y claves foráneas, removemos nombres y urls de texto largo)
fact_global_charts = df_global_charts[[
    'date_key', 'rank_position', 'song_id', 'artist_id', 'playcount_daily', 'listeners_daily'
]].copy()

print("✅ Datos normalizados con éxito.")
print(f"📊 Filas en Dim_Songs: {len(dim_songs)}")
print(f"📊 Filas en Dim_Artists: {len(dim_artists)}")
print(f"📊 Filas en Fact_Global_Charts: {len(fact_global_charts)}")

# 4. Exportar a archivos CSV locales en el entorno de Colab
dim_songs.to_csv('dim_songs.csv', index=False, encoding='utf-8')
dim_artists.to_csv('dim_artists.csv', index=False, encoding='utf-8')
fact_global_charts.to_csv('fact_global_charts.csv', index=False, encoding='utf-8')

print("\n📥 Descargando archivos a tu computadora...")
# 5. Forzar la descarga automática de los 3 archivos en tu navegador
files.download('dim_songs.csv')
files.download('dim_artists.csv')
files.download('fact_global_charts.csv')

⚙️ Iniciando normalización de datos para el Modelo Dimensional...
✅ Datos normalizados con éxito.
📊 Filas en Dim_Songs: 50
📊 Filas en Dim_Artists: 23
📊 Filas en Fact_Global_Charts: 50

📥 Descargando archivos a tu computadora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>